# Eksperyment: HistGradientBoosting vs Random Forest

## Cel
Sprawdzić, czy gradient boosting (HistGradientBoosting z sklearn) pobije Random Forest na tych
samych cechach i danych. Dwa warianty HGB: numeryczny (jak RF) oraz z **natywnymi kategoriami**
(surface/tourney_level jako nominalne -- główna przewaga HGB).

In [1]:
import sys, io, contextlib, runpy
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import os
# --- Colab: pobiera projekt z GitHuba; lokalnie ten blok jest pomijany ---
# PO UTWORZENIU repo podmien adres ponizej na swoj:
_REPO = "https://github.com/StanislawKarwala/TennisPredictionModel.git"
if "google.colab" in sys.modules and not Path("../src/tennis_model.py").exists():
    import subprocess
    subprocess.run(["pip", "install", "-q", "xgboost"])
    subprocess.run(["git", "clone", "-q", _REPO, "/content/tenis"])
    os.chdir("/content/tenis/notebooks")
sys.path.insert(0, str(Path("../src").resolve()))
from sklearn.ensemble import HistGradientBoostingClassifier
from tennis_model_hgb import tune_and_eval_hgb, print_row

## 1. Reuse baseline pipeline (RF już policzony)
Baseline zwraca gotowy RF + dane CV i test. Liczymy jakość prawdopodobieństw RF do porównania.

In [2]:
BASE = Path("../src/tennis_model.py").resolve()
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    ns = runpy.run_path(str(BASE))
symmetrize_data = ns["symmetrize_data"]
compute_symmetric_match_evaluation = ns["compute_symmetric_match_evaluation"]
evaluate_calibration_quality = ns["evaluate_calibration_quality"]
baseline_search = ns["search"]
RANDOM_STATE = ns["RANDOM_STATE"]
base_features = list(ns["features"]); cols_base = list(ns["cols_base"])
df_train_raw = ns["df_train_raw"].copy(); df_val_raw = ns["df_val_raw"].copy(); df_test_raw = ns["df_test_raw"].copy()
baseline_val_acc = float(ns["val_acc"]); baseline_test_acc = float(ns["test_acc"]); baseline_match_acc = float(ns["match_accuracy"])
print(f"Baseline: val={baseline_val_acc:.4f}  test={baseline_test_acc:.4f}  match={baseline_match_acc:.4f}  (cech: {len(base_features)})")
features = base_features
X_train_cv, y_train_cv = ns["X_train_cv"], ns["y_train_cv"]
X_val, y_val = ns["X_val"], ns["y_val"]; X_test, y_test = ns["X_test"], ns["y_test"]
test_data = ns["test_data"]
best_rf = ns["best_rf"]
rf_proba = best_rf.predict_proba(X_test)[:, 1]
rf_q = evaluate_calibration_quality(y_test.to_numpy(), rf_proba)
print(f"RF: match={baseline_match_acc:.4f}  Brier={rf_q['brier_score']:.4f}")

Baseline: val=0.6106  test=0.6604  match=0.6566  (cech: 40)
RF: match=0.6566  Brier=0.2174


## 2. Strojenie HGB (RandomizedSearchCV, neg_log_loss) -- dwa warianty
`tune_and_eval_hgb` stroi HGB na TimeSeriesSplit i ocenia val/test/match + kalibrację.

In [3]:
common = dict(features=features, X_train_cv=X_train_cv, y_train_cv=y_train_cv, df_train_raw=df_train_raw,
    symmetrize_data=symmetrize_data, X_val=X_val, y_val=y_val, X_test=X_test, y_test=y_test, test_data=test_data,
    compute_symmetric_match_evaluation=compute_symmetric_match_evaluation,
    evaluate_calibration_quality=evaluate_calibration_quality, RANDOM_STATE=RANDOM_STATE)
hgb_num = tune_and_eval_hgb(label="HGB (numeric)", categorical_features=None, **common)
cat_cols = [c for c in ("surface", "tourney_level") if c in features]
hgb_cat = tune_and_eval_hgb(label="HGB (kategorie)", categorical_features=cat_cols, **common)
print("gotowe")


Strojenie HGB (numeric) (RandomizedSearchCV, neg_log_loss)...


  najlepsze HP: {'min_samples_leaf': 120, 'max_leaf_nodes': 63, 'max_iter': 200, 'max_features': 0.6, 'max_depth': 8, 'learning_rate': 0.02, 'l2_regularization': 0.1}
  CV: neg_log_loss=-0.6409 | accuracy=0.6325 | roc_auc=0.6824



Strojenie HGB (kategorie) (RandomizedSearchCV, neg_log_loss)...


  najlepsze HP: {'min_samples_leaf': 120, 'max_leaf_nodes': 63, 'max_iter': 200, 'max_features': 0.6, 'max_depth': 8, 'learning_rate': 0.02, 'l2_regularization': 1.0}
  CV: neg_log_loss=-0.6406 | accuracy=0.6295 | roc_auc=0.6827


gotowe


## 3. Porównanie RF vs HGB

In [4]:
print(f"{'model':<16}{'val':>9}{'test':>9}{'match':>9}{'Brier':>9}{'logloss':>9}")
print(f"{'RandomForest':<16}{baseline_val_acc:>9.4f}{baseline_test_acc:>9.4f}{baseline_match_acc:>9.4f}{rf_q['brier_score']:>9.4f}{rf_q['log_loss']:>9.4f}")
for r in (hgb_num, hgb_cat):
    print(f"{r['label']:<16}{r['val_acc']:>9.4f}{r['test_acc']:>9.4f}{r['match_acc']:>9.4f}{r['brier']:>9.4f}{r['logloss']:>9.4f}")
for r in (hgb_num, hgb_cat):
    print(f"DELTA ({r['label']} - RF): match={r['match_acc']-baseline_match_acc:+.4f}")

model                 val     test    match    Brier  logloss
RandomForest       0.6106   0.6604   0.6566   0.2174   0.6246
HGB (numeric)      0.5974   0.6575   0.6547   0.2162   0.6208
HGB (kategorie)    0.6011   0.6557   0.6585   0.2156   0.6197
DELTA (HGB (numeric) - RF): match=-0.0019
DELTA (HGB (kategorie) - RF): match=+0.0019


## Wnioski
HistGradientBoosting wyszedł mniej więcej na równi z Random Forest — raz odrobinę gorzej, raz odrobinę lepiej, wszystko w granicach szumu — a kalibrację miał słabszą. Przy około 3500 meczach treningowych boosting nie ma z czego rozwinąć przewagi i wybiera mocno regularyzowane ustawienia. Dlatego zostaję przy Random Forest. Ten sam test powtórzyłem na dużo większym zbiorze (notebook multiseason, ~123 tys. próbek) — wynik dalej ~65%.